# 41 · Tratamento de outliers na tabela master

Aplica as sugestões do notebook 40 para preparar a tabela para o k-means. **Nenhuma coluna original
é alterada e nenhuma linha é removida**: o tratamento entra em colunas novas, com sufixo.

1. **Outliers reais → limite no percentil 99** (sufixo `_cap`): `minutos_90d`, `treinos_90d`,
   `aderencia_meta`. O limite é o p99 dos assinantes ativos no corte e vale para todas as linhas.
2. **Cauda longa → logaritmo** (`log(1 + x)`, sufixo `_log`): `dias_sem_treinar`, `duracao_dias`,
   `n_cobrancas_aprovadas` e treinos de corrida, funcional, caminhada, força e yoga.
3. **Concentradas em zero → indicador** (sim ou não): `pedala_90d`, `treinou_acompanhado_90d`,
   `abriu_chamado`, `tem_relogio`.
4. **Mais posts do que treinos**: valor mantido, sem coluna nova.

O notebook lê `tabela_master.parquet` (notebook 39), acrescenta as colunas e grava de volta. Pode ser
rodado mais de uma vez: as colunas tratadas são recriadas a cada execução.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
P = Path("../dados/processados"); S = Path("../saidas")

m = pd.read_parquet(P / "tabela_master.parquet")
print("tabela master lida:", m.shape)
CAP = ["minutos_90d", "treinos_90d", "aderencia_meta"]
LOG = ["dias_sem_treinar", "duracao_dias", "n_cobrancas_aprovadas", "treinos_corrida", "treinos_funcional", "treinos_caminhada", "treinos_forca", "treinos_yoga"]
IND = ["pedala_90d", "treinou_acompanhado_90d", "abriu_chamado", "tem_relogio"]
novas = [c + "_cap" for c in CAP] + [c + "_log" for c in LOG] + IND
ja = [c for c in novas if c in m.columns]
m = m.drop(columns=ja)
print("colunas tratadas que já existiam e serão recriadas:", len(ja), "| colunas originais:", m.shape[1])
orig = m.copy()
at = m[m.ativa_no_corte]
print("população de referência dos limites (assinantes ativos no corte):", len(at))

tabela master lida: (63929, 80)
colunas tratadas que já existiam e serão recriadas: 0 | colunas originais: 80
população de referência dos limites (assinantes ativos no corte): 14753


## 1. Limite no percentil 99

In [2]:
reg = []
for c in CAP:
    lim = float(at[c].astype(float).quantile(.99))
    s = m[c].astype(float)
    m[c + "_cap"] = s.clip(upper=lim)
    n = int((s > lim).sum()); na = int((at[c].astype(float) > lim).sum())
    reg.append({"coluna": c, "nova": c + "_cap", "limite (p99 dos ativos)": lim, "linhas limitadas": n, "entre ativos": na, "máximo antes": s.max(), "máximo depois": m[c + "_cap"].max(), "preenchidas": int(s.notna().sum())})
r1 = pd.DataFrame(reg).set_index("coluna"); print(r1)

                              nova  limite (p99 dos ativos)  linhas limitadas  entre ativos  máximo antes  máximo depois  preenchidas
coluna                                                                                                                               
minutos_90d        minutos_90d_cap                 4,250.96               190           148     11,559.00       4,250.96        54470
treinos_90d        treinos_90d_cap                    84.00               166           139        162.00          84.00        54470
aderencia_meta  aderencia_meta_cap                     1.86               161           133          3.27           1.86        48278


## 2. Logaritmo nas colunas de cauda longa

In [3]:
reg = []
for c in LOG:
    s = m[c].astype(float)
    assert (s.dropna() >= 0).all()
    m[c + "_log"] = np.log1p(s).round(4)
    reg.append({"coluna": c, "nova": c + "_log", "preenchidas": int(s.notna().sum()), "assimetria antes (ativos)": at[c].astype(float).skew(), "assimetria depois (ativos)": np.log1p(at[c].astype(float)).skew(),
                "máximo antes": s.max(), "máximo depois": m[c + "_log"].max()})
r2 = pd.DataFrame(reg).set_index("coluna"); print(r2)

                                            nova  preenchidas  assimetria antes (ativos)  assimetria depois (ativos)  máximo antes  máximo depois
coluna                                                                                                                                           
dias_sem_treinar            dias_sem_treinar_log        36164                       4.41                        0.98        272.00           5.61
duracao_dias                    duracao_dias_log        32583                       2.13                       -0.90      2,213.00           7.70
n_cobrancas_aprovadas  n_cobrancas_aprovadas_log        32583                       2.07                        1.10         28.00           3.37
treinos_corrida              treinos_corrida_log        54470                       2.14                        0.68        100.00           4.62
treinos_funcional          treinos_funcional_log        54470                       1.94                        0.07        

## 3. Indicadores para as colunas concentradas em zero

In [4]:
m["pedala_90d"] = (m.treinos_pedal > 0).astype("boolean").where(m.treinos_pedal.notna())
m["treinou_acompanhado_90d"] = (m.treinos_acompanhados_90d > 0).astype("boolean").where(m.treinos_acompanhados_90d.notna())
m["abriu_chamado"] = (m.n_chamados > 0)
m["tem_relogio"] = (m.relogio != "nenhum")
BASE = {"pedala_90d": "treinos_pedal", "treinou_acompanhado_90d": "treinos_acompanhados_90d", "abriu_chamado": "n_chamados", "tem_relogio": "relogio"}
r3 = pd.DataFrame([{"nova": k, "origem": v, "preenchidas": int(m[k].notna().sum()), "sim": int(m[k].sum()), "% sim (preenchidas)": float(m[k].mean() * 100),
                    "sim entre ativos": int(m.loc[m.ativa_no_corte, k].sum()), "% sim (ativos)": float(m.loc[m.ativa_no_corte, k].mean() * 100)} for k, v in BASE.items()]).set_index("nova")
print(r3)
print()
print("mais posts do que treinos em 90 dias (mantido como está):", int((m.posts_90d.notna() & m.treinos_90d.notna() & (m.posts_90d > m.treinos_90d)).sum()), "contas")

                                           origem  preenchidas    sim  % sim (preenchidas)  sim entre ativos  % sim (ativos)
nova                                                                                                                        
pedala_90d                          treinos_pedal        54470   2739                 5.03              1383            9.37
treinou_acompanhado_90d  treinos_acompanhados_90d        54470   1715                 3.15              1080            7.32
abriu_chamado                          n_chamados        63929   3947                 6.17              2025           13.73
tem_relogio                               relogio        63929  24414                38.19              5635           38.20

mais posts do que treinos em 90 dias (mantido como está): 1213 contas


## 4. Conferências

In [5]:
print("linhas:", len(m), "(antes:", len(orig), ") | colunas:", orig.shape[1], "->", m.shape[1], "| novas:", m.shape[1] - orig.shape[1])
iguais = all(m[c].equals(orig[c]) for c in orig.columns)
print("colunas originais intactas:", iguais); assert iguais and len(m) == len(orig) and m.user_id.is_unique
for c in CAP:
    assert (m[c + "_cap"].dropna() <= m[c].astype(float).dropna()).all() and m[c + "_cap"].isna().equals(m[c].isna())
for c in LOG:
    assert m[c + "_log"].isna().equals(m[c].isna())
print("vazios preservados em todas as colunas novas: True")
from importlib import import_module
def fora3(s):
    s = s.dropna().astype(float); q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return int(((s > q3 + 3 * i) | (s < q1 - 3 * i)).sum())
a2 = m[m.ativa_no_corte]
comp = pd.DataFrame([{"coluna": c, "fora de 3·IQR antes": fora3(a2[c]), "tratada": t, "fora de 3·IQR depois": fora3(a2[t]), "assimetria antes": a2[c].astype(float).skew(), "assimetria depois": a2[t].astype(float).skew()}
                     for c, t in [(c, c + "_cap") for c in CAP] + [(c, c + "_log") for c in LOG]]).set_index("coluna")
print(); print("assinantes ativos: extremos antes e depois"); print(comp)

linhas: 63929 (antes: 63929 ) | colunas: 80 -> 95 | novas: 15


colunas originais intactas: True
vazios preservados em todas as colunas novas: True

assinantes ativos: extremos antes e depois
                       fora de 3·IQR antes                    tratada  fora de 3·IQR depois  assimetria antes  assimetria depois
coluna                                                                                                                          
minutos_90d                             66            minutos_90d_cap                     0              1.70               1.01
treinos_90d                             15            treinos_90d_cap                     0              0.98               0.77
aderencia_meta                          22         aderencia_meta_cap                     0              0.84               0.52
dias_sem_treinar                      1557       dias_sem_treinar_log                     0              4.41               0.99
duracao_dias                           530           duracao_dias_log                    32       

## 5. Dicionário e gravação

In [6]:
dic = pd.read_csv(S / "tabela_master_dicionario.csv")
dic = dic[~dic.coluna.isin(novas)][["coluna", "grupo", "descricao", "fonte"]]
lim = r1["limite (p99 dos ativos)"]
extra = [(c + "_cap", "tratada para k-means", f"{c} limitado em {lim[c]:,.2f} (percentil 99 dos assinantes ativos no corte)".replace(",", "X").replace(".", ",").replace("X", "."), c) for c in CAP]
extra += [(c + "_log", "tratada para k-means", f"log(1 + {c}), para reduzir a cauda longa", c) for c in LOG]
extra += [("pedala_90d", "tratada para k-means", "Fez algum treino de pedal na janela (substitui treinos_pedal, 91% de zeros entre ativos)", "treinos_pedal"),
          ("treinou_acompanhado_90d", "tratada para k-means", "Fez algum treino acompanhado na janela (substitui treinos_acompanhados_90d, 93% de zeros)", "treinos_acompanhados_90d"),
          ("abriu_chamado", "tratada para k-means", "Abriu algum chamado de suporte até a referência (substitui n_chamados, 86% de zeros)", "n_chamados"),
          ("tem_relogio", "tratada para k-means", "Tem relógio no cadastro (substitui pct_treinos_relogio, 62% de zeros). Situação na exportação", "relogio")]
dic = pd.concat([dic, pd.DataFrame(extra, columns=["coluna", "grupo", "descricao", "fonte"])], ignore_index=True)
assert list(dic.coluna) == list(m.columns), "dicionário e tabela fora de ordem"
dic["tipo"] = [str(m[c].dtype) for c in dic.coluna]
dic["preenchidas"] = [int(m[c].notna().sum()) for c in dic.coluna]
dic["% preenchido"] = (dic.preenchidas / len(m) * 100).round(1)
dic["% preenchido (ativos)"] = [round(m.loc[m.ativa_no_corte, c].notna().mean() * 100, 1) for c in dic.coluna]
m.to_parquet(P / "tabela_master.parquet", index=False)
m.to_csv(S / "tabela_master.csv", index=False, encoding="utf-8-sig")
dic.to_csv(S / "tabela_master_dicionario.csv", index=False, encoding="utf-8-sig")
pd.concat({"limite p99": r1.reset_index(), "logaritmo": r2.reset_index(), "indicador": r3.reset_index()}).to_csv(S / "tabela_master_tratamento_outliers.csv", encoding="utf-8-sig")
print("gravado:", m.shape)
print(dic[dic.grupo == "tratada para k-means"][["coluna", "descricao", "% preenchido", "% preenchido (ativos)"]].to_string())

gravado: (63929, 95)
                       coluna                                                                                      descricao  % preenchido  % preenchido (ativos)
80            minutos_90d_cap                 minutos_90d limitado em 4.250,96 (percentil 99 dos assinantes ativos no corte)         85.20                 100.00
81            treinos_90d_cap                    treinos_90d limitado em 84,00 (percentil 99 dos assinantes ativos no corte)         85.20                 100.00
82         aderencia_meta_cap                  aderencia_meta limitado em 1,86 (percentil 99 dos assinantes ativos no corte)         75.50                  89.90
83       dias_sem_treinar_log                                          log(1 + dias_sem_treinar), para reduzir a cauda longa         56.60                  98.80
84           duracao_dias_log                                              log(1 + duracao_dias), para reduzir a cauda longa         51.00                 100.00
85  n_c